# Crivo — segunda rodada JS/TS

Currículo com 3.600 exemplos de código autorais e casos de teste. Variantes cobrem 30 famílias; não comprovam nível sênior.

Ative GPU e execute todas as células. Use uma execução nova (EXECUCAO_ID vazio). Para aproveitar seu treino anterior, preencha MODELO_INICIAL com o caminho completo de sua pasta candidato no Drive; deixe vazio para iniciar dos pesos gerais próprios.

Esta rodada mantém 2,6 milhões de parâmetros e usa 6.000 passos. Salva o resultado em uma nova pasta.


In [ ]:
# 1. Conferir a GPU antes de instalar ou treinar
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU indisponível. Ative GPU T4 no menu Ambiente de execução e execute novamente.")
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU)
print("PyTorch:", torch.__version__)
print("Memória da GPU (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## Código reproduzível
O código de treinamento está fixado no commit `072b48d93dc49efaf02bebaae4365eb0fef2776d`. Assim, uma atualização da branch não muda o treinador durante uma retomada. O notebook utiliza os pesos próprios presentes nessa revisão.

In [ ]:
# 2. Baixar a revisão fixada, sem sobrescrever outro checkout
import subprocess
import sys
from pathlib import Path

REPOSITORIO = "https://github.com/HROSONE/CRIVO.git"
REVISAO = "072b48d93dc49efaf02bebaae4365eb0fef2776d"
ROOT = Path("/content/CRIVO-programacao-v2")
if ROOT.exists():
    if not (ROOT / ".git").is_dir():
        raise RuntimeError(f"A pasta {ROOT} já existe e não é um checkout Git. Use outra pasta.")
    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
    if head != REVISAO:
        raise RuntimeError("Checkout de outra revisão encontrado. Use outra pasta para preservar o trabalho existente.")
else:
    ROOT.mkdir(parents=True)
    subprocess.run(["git", "init", "-q"], cwd=ROOT, check=True)
    subprocess.run(["git", "remote", "add", "origin", REPOSITORIO], cwd=ROOT, check=True)
    subprocess.run(["git", "fetch", "--depth", "1", "origin", REVISAO], cwd=ROOT, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
if subprocess.run(["git", "diff", "--quiet", "HEAD"], cwd=ROOT).returncode:
    raise RuntimeError("Há arquivos modificados no checkout. Preserve-os e use outra pasta para o treino reproduzível.")
print("Código:", ROOT, "\nRevisão:", REVISAO)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# 3. Dependências: preservar o PyTorch com CUDA que já vem no Colab
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "tokenizers>=0.20,<1", "numpy>=1.24,<3", "quickjs==1.19.4"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "nodejs", "npm", "bubblewrap"], check=True)
subprocess.run(["npm", "install", "--prefix", "/tmp/crivo-ts", "typescript@5.9.3"], check=True)
TSC = Path("/tmp/crivo-ts/node_modules/typescript/lib/tsc.js")
assert TSC.is_file(), "Compilador TypeScript ausente"
subprocess.run(["node", "--version"], check=True)
subprocess.run(["node", str(TSC), "--version"], check=True)

In [ ]:
# Verificar JS/TS antes de gastar GPU; resultados QuickJS são identificados
subprocess.run([sys.executable, str(ROOT / "scripts/verificar_ambiente_programacao.py"),
                "--tsc", str(TSC)], cwd=ROOT, check=True)
print("QuickJS não fornece APIs Node ou de navegador. O candidato segue experimental.")


In [ ]:
# 4. Configuração — use os mesmos valores ao retomar
PERFIL = "atual" #@param ["atual", "10m", "30m"]
PASSOS = 6000 #@param {type:"integer"}
LOTE = 4 #@param {type:"integer"}
THREADS = 2 #@param {type:"integer"}
TEMPO_MAXIMO_SEGUNDOS = 7200 #@param {type:"integer"}
PASTA_DRIVE = "/content/drive/MyDrive/CRIVO/treinos" #@param {type:"string"}
MODELO_INICIAL = "" #@param {type:"string"}
EXECUCAO_ID = "" #@param {type:"string"}

from scripts.escalar_programacao import PERFIS, estimar
if PERFIL not in PERFIS or min(PASSOS, LOTE, THREADS, TEMPO_MAXIMO_SEGUNDOS) < 1:
    raise ValueError("Perfil inválido ou contagem não positiva")
config_modelo = PERFIS[PERFIL]
print(estimar(PERFIL))
if PERFIL != "atual":
    print("Este perfil inicia um modelo maior do zero: pré-treino e diálogo são etapas separadas.")
    print("O corpus pequeno não é suficiente para explorar essa capacidade; mantenha 'atual' no primeiro piloto.")

In [ ]:
# 5. Conectar o Google Drive — o Colab pedirá sua autorização
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 6. Criar execução ou recuperar uma execução anterior
import hashlib
import json
import re
import shutil
import uuid
from datetime import datetime, timezone
import tokenizers

if not EXECUCAO_ID:
    EXECUCAO_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]
if not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", EXECUCAO_ID):
    raise ValueError("EXECUCAO_ID deve conter somente letras, números, hífen ou sublinhado")
base_drive = Path(PASTA_DRIVE).resolve()
my_drive = Path("/content/drive/MyDrive").resolve()
if my_drive not in base_drive.parents and base_drive != my_drive:
    raise ValueError("Escolha uma pasta dentro de /content/drive/MyDrive para preservar os arquivos")
RUN = base_drive / EXECUCAO_ID
LOCAL = Path("/content/crivo-corpus-v2") / EXECUCAO_ID
INICIAL = Path(MODELO_INICIAL).expanduser().resolve() if MODELO_INICIAL.strip() else ROOT / "artefatos/linguagem_profunda"
if MODELO_INICIAL.strip() and PERFIL != "atual":
    raise ValueError("MODELO_INICIAL é compatível somente com o perfil atual")
origem_hashes = {}
if PERFIL == "atual":
    for nome in ("pesos.pt", "tokenizer.json", "relatorio.json"):
        arquivo = INICIAL / nome
        if not arquivo.is_file():
            raise FileNotFoundError(f"Modelo inicial incompleto: {arquivo}")
        origem_hashes[nome] = hashlib.sha256(arquivo.read_bytes()).hexdigest()
    origem = json.loads((INICIAL / "relatorio.json").read_text())
    if any(origem.get("config", {}).get(k) != v for k, v in config_modelo.items()):
        raise ValueError("Arquitetura do modelo inicial diferente do perfil atual")
    if origem.get("pesos_sha256") != origem_hashes["pesos.pt"]:
        raise ValueError("Pesos iniciais não correspondem ao relatório")
    tokenizer_padrao = hashlib.sha256((ROOT / "artefatos/linguagem_profunda/tokenizer.json").read_bytes()).hexdigest()
    if origem_hashes["tokenizer.json"] != tokenizer_padrao:
        raise ValueError("Tokenizer inicial diferente do corpus desta rodada")
config_execucao = dict(modelo_inicial=str(INICIAL), origem_hashes=origem_hashes, revisao=REVISAO, perfil=PERFIL, passos=PASSOS, lote=LOTE,
                       threads=THREADS, config_modelo=config_modelo,
                       torch=str(torch.__version__), tokenizers=tokenizers.__version__, gpu=GPU)
meta = RUN / "execucao.json"
if meta.exists():
    if json.loads(meta.read_text()) != config_execucao:
        raise RuntimeError("Configuração/ambiente diferente da execução salva. Restaure os valores originais ou crie uma execução nova.")
    print("Execução existente: checkpoints serão retomados.")
else:
    if RUN.exists() and any(RUN.iterdir()):
        raise RuntimeError("Pasta já contém arquivos sem identificação. Use um EXECUCAO_ID novo.")
    RUN.mkdir(parents=True, exist_ok=True)
    meta.write_text(json.dumps(config_execucao, ensure_ascii=False, indent=2))
print("GUARDE ESTE EXECUCAO_ID:", EXECUCAO_ID)
print("Arquivos permanentes:", RUN)
print("Para retomar: preencha EXECUCAO_ID e mantenha os parâmetros acima.")

## Preparar e treinar
A leitura do corpus ocorre no disco local para evitar I/O lento do Drive. Uma cópia permanente permite reconstruir esse disco após uma desconexão. Pesos, estado de Adam/RNG e progresso são salvos diretamente na pasta da execução no Drive.

O log mostra progresso a cada 20 passos. O limite de tempo interrompe o processo; execute novamente com o mesmo identificador para continuar do último checkpoint salvo. No perfil maior, o limite se aplica separadamente a cada etapa.

In [ ]:
# 7. Executor com progresso visível, log no Drive e limite de tempo
import threading

def executar(script, argumentos):
    cmd = [sys.executable, "-u", str(ROOT / "scripts" / script)] + [str(a) for a in argumentos]
    print("Executando:", script, flush=True)
    interrompido = threading.Event()
    with (RUN / "console.log").open("a", encoding="utf-8") as log:
        log.write("\nCOMANDO " + json.dumps(cmd, ensure_ascii=False) + "\n")
        log.flush()
        processo = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1)
        def parar():
            if processo.poll() is None:
                interrompido.set()
                processo.kill()
        timer = threading.Timer(TEMPO_MAXIMO_SEGUNDOS, parar)
        timer.daemon = True
        timer.start()
        try:
            for linha in processo.stdout:
                print(linha, end="", flush=True)
                log.write(linha)
                log.flush()
            retorno = processo.wait()
        except BaseException:
            if processo.poll() is None:
                processo.kill()
            processo.wait()
            raise
        finally:
            timer.cancel()
            processo.stdout.close()
        if interrompido.is_set():
            raise RuntimeError("Tempo limite atingido. Retome com o mesmo EXECUCAO_ID; os checkpoints já salvos permanecem no Drive.")
        if retorno:
            raise subprocess.CalledProcessError(retorno, cmd)

In [ ]:
# 8. Preparar corpus ou restaurar a cópia permanente
corpus_drive = RUN / "corpus"
if not corpus_drive.exists():
    if LOCAL.exists():
        raise RuntimeError("Preparação local incompleta encontrada. Use um novo EXECUCAO_ID ou inspecione essa pasta antes de repetir.")
    args = ["--saida", LOCAL, "--contexto", config_modelo["contexto"],
            "--vocabulario", config_modelo["vocabulario"]]
    if PERFIL == "atual":
        args += ["--tokenizer", ROOT / "artefatos/linguagem_profunda/tokenizer.json"]
    executar("preparar_programacao.py", args)
    temporario = RUN / "corpus-em-copia"
    if temporario.exists():
        raise RuntimeError("Cópia anterior interrompida. Inspecione corpus-em-copia antes de continuar.")
    shutil.copytree(LOCAL, temporario)
    temporario.rename(corpus_drive)
elif not LOCAL.exists():
    LOCAL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(corpus_drive, LOCAL)

# O treinador também verifica o manifesto; conferir já permite detectar cópia incompleta.
from scripts.treinar_linguagem_profunda import Corpus
corpus = Corpus(LOCAL, config_modelo["contexto"])
print(json.dumps(corpus.manifesto["particoes"], ensure_ascii=False, indent=2))

In [ ]:
# 9. Treinar em CUDA ou retomar checkpoints; etapas concluídas são preservadas
BASE = ["--corpus", LOCAL, "--passos", PASSOS, "--lote", LOTE, "--threads", THREADS,
        "--dimensao", config_modelo["dimensao"], "--camadas", config_modelo["camadas"],
        "--cabecas", config_modelo["cabecas"], "--contexto", config_modelo["contexto"],
        "--avaliar-a-cada", min(200, PASSOS), "--salvar-a-cada", min(100, PASSOS),
        "--dispositivo", "cuda"]

def treinar_etapa(pasta, fase, inicial=None, ajustar=False):
    relatorio = pasta / "relatorio.json"
    pesos = pasta / "pesos.pt"
    if (pasta / "checkpoint.pt").exists() and relatorio.exists() and pesos.exists():
        salvo = json.loads(relatorio.read_text())
        if salvo.get("passo") == PASSOS and salvo.get("pesos_sha256") == hashlib.sha256(pesos.read_bytes()).hexdigest():
            print("Etapa já concluída e pesos conferidos:", fase)
            return  # Preservar pesos, relatório original e avaliação existente.
    args = BASE + ["--saida", pasta, "--fase", fase]
    if fase == "dialogo":
        args += ["--lr", ".0001"]
    if (pasta / "checkpoint.pt").exists():
        # O treinador verifica dados/código/configuração antes de retomar.
        args += ["--retomar"]
    elif inicial is not None:
        args += ["--inicial", inicial]
        if ajustar:
            args += ["--ajustar-proprio"]
    executar("treinar_linguagem_profunda.py", args)

if PERFIL == "atual":
    inicial = INICIAL
else:
    inicial = RUN / "pretreino"
    treinar_etapa(inicial, "linguagem")
CANDIDATO = RUN / "candidato"
treinar_etapa(CANDIDATO, "dialogo", inicial, ajustar=(PERFIL == "atual"))
relatorio_treino = json.loads((CANDIDATO / "relatorio.json").read_text())
print("Etapa concluída:", relatorio_treino["passo"], "passos")
print("Parâmetros reais:", relatorio_treino["parametros"])
print("Pesos salvos:", CANDIDATO / "pesos.pt")

## Avaliar o código
A avaliação registra compilação, término, execução funcional quando disponível e um reparo limitado pelo contexto. Se o Colab bloquear os namespaces do sandbox, a execução funcional fica indisponível; código candidato não é executado sem isolamento.

Não use as respostas de teste como dados de treino nem ajuste o corpus para copiar essas saídas. Este conjunto é pequeno e público. Para promover o modelo serão necessárias mais famílias e avaliação independente.

In [ ]:
# 10. Avaliar uma vez; uma avaliação existente é exibida novamente sem sobrescrever
AVALIACAO = RUN / "avaliacao.json"
if not AVALIACAO.exists():
    executar("avaliar_programacao.py", ["--modelo", CANDIDATO, "--saida", AVALIACAO,
                                       "--tsc", TSC, "--reparos", 1])
resultado = json.loads(AVALIACAO.read_text())
from programacao_neural import digest
if resultado["pesos_sha256"] != digest(CANDIDATO / "pesos.pt"):
    raise RuntimeError("A avaliação existente pertence a outros pesos. Preserve-a e avalie o novo candidato em outro arquivo.")
print(json.dumps({k: resultado[k] for k in ("linguagens", "isolamento", "gate")},
                 ensure_ascii=False, indent=2))

In [ ]:
# 11. Resumo legível
from IPython.display import Markdown, display
linhas = ["| Linguagem | Compilam | Completas | Execuções funcionais | Pass@1 |",
          "|---|---:|---:|---:|---:|"]
for linguagem, m in resultado["linguagens"].items():
    taxa = "Indisponível" if m["pass_at_1"] is None else f"{m['pass_at_1']:.1%}"
    linhas.append(f"| {linguagem} | {m['compilam']}/{m['total']} | {m['completas']}/{m['total']} | {m['executadas']}/{m['total']} | {taxa} |")
display(Markdown("\n".join(linhas)))
print("Gate aprovado:", resultado["gate"]["aprovado"])
for motivo in resultado["gate"]["motivos"]:
    print("•", motivo)
print("Relatório:", AVALIACAO)
print("Perda menor, sozinha, não comprova código correto. Nenhuma ativação automática foi realizada.")

## Teste opcional de uma geração
Execute a próxima célula somente depois do treino. Ela mostra código experimental e **não executa** a saída.

In [ ]:
MENSAGEM = "Em javascript, implemente function resolver(n) para dobrar n. Retorne somente código." #@param {type:"string"}
from programacao_neural import GeradorProgramacao
gerador = GeradorProgramacao(CANDIDATO)
print(json.dumps(gerador.gerar(MENSAGEM), ensure_ascii=False, indent=2))

## Arquivos para compartilhar
O Drive já contém `candidato/checkpoint.pt` (retomada com Adam/RNG), `candidato/pesos.pt`, tokenizer, corpus, logs e relatórios. O ZIP abaixo inclui pesos de inferência e relatórios; o checkpoint completo continua no Drive.

Publique código e relatórios no GitHub. Preserve seus checkpoints no Drive. O próximo ciclo deve ampliar e equilibrar exemplos de código correto/testado antes de aumentar o modelo.

In [ ]:
# 12. Exportar o candidato e relatórios para um ZIP no Drive
import zipfile
ZIP = RUN / ("crivo-programacao-" + EXECUCAO_ID + ".zip")
arquivos = [CANDIDATO / "pesos.pt", CANDIDATO / "tokenizer.json",
            CANDIDATO / "relatorio.json", RUN / "execucao.json", AVALIACAO,
            corpus_drive / "manifesto.json"]
with zipfile.ZipFile(ZIP, "w", compression=zipfile.ZIP_DEFLATED) as pacote:
    for arquivo in arquivos:
        pacote.write(arquivo, arquivo.relative_to(RUN))
print("ZIP salvo:", ZIP)
# Para baixar no navegador, remova o # abaixo:
# from google.colab import files; files.download(str(ZIP))